# Week 2 excercise
Technical questions assistant with multi modal capabilities (chat, audio and image).
Also uses a tool to check if the provided code snippet is valid python code (the tool is a python sintax checker).

In [41]:
# imports
import os
from dotenv import load_dotenv
from openai import OpenAI
import base64
from io import BytesIO
from PIL import Image
import json
import ast
import gradio as gr

In [42]:
# constants
MODEL_GPT = 'gpt-4.1-mini'
MODEL_IMAGE = 'dall-e-3'
MODEL_TTS = 'gpt-4o-mini-tts'


In [43]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("OpenAI API Key was correctly setup.")
else:
    print("OpenAPI API Key is missing.")

openai = OpenAI()

technical_explainer_system_prompt = """
You will be asked code technical questions.
You can call a tool called check_sintax to check if python code is valid.
You will be provided with python code snippets,
explain every line and what it does.
Respond in markdown without html tags.
"""


OpenAI API Key was correctly setup.


In [44]:
# image generation
def artist(code_snippet):
    image_response = openai.images.generate(
            model="dall-e-3",
            prompt=f"An image of code snippet {code_snippet}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))


In [45]:
# audio generation
def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",
      input=message
    )
    return response.content


In [46]:
# check if code snippet is valid python code
def is_valid_python(code_snippet):
    try:
        ast.parse(code_snippet)
        return "Success! The provided code snippet is valid python code."
    except SyntaxError:
        return "Error! The provided snippet is not valid python code!"


In [47]:
# tool function definition in json
valid_python_code_function = {
    "name": "check_sintax",
    "description": "Check if the provided python code snippet has correct sintax.",
    "parameters": {
        "type": "object",
        "properties": {
            "snippet": {
                "type": "string",
                "description": "The python code snippet",
            },
        },
        "required": ["snippet"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": valid_python_code_function}]
tools


[{'type': 'function',
  'function': {'name': 'check_sintax',
   'description': 'Check if the provided python code snippet has correct sintax.',
   'parameters': {'type': 'object',
    'properties': {'snippet': {'type': 'string',
      'description': 'The python code snippet'}},
    'required': ['snippet'],
    'additionalProperties': False}}}]

In [51]:
# tool call
def handle_tool_calls_and_return_snippets(message):
    responses = []
    snippets = []
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        snippet = arguments.get('snippet')
        snippets.append(snippet)
        valid_python_code = is_valid_python(snippet)
        responses.append({
            "role": "tool",
            "content": valid_python_code,
            "tool_call_id": tool_call.id
        })
    return responses, snippets


In [52]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": technical_explainer_system_prompt}] + history
    response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)
    snippets = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, snippets = handle_tool_calls_and_return_snippets(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if snippets:
        image = artist(snippets[0])
    
    return history, voice, image


In [53]:
# Callbacks (along with the chat() function above)
def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

    # Hooking up events to callbacks
    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.
